# Extract emotion steering vectors (Qwen2.5-3B)

Runs `emotion.extract_vectors`: loads Qwen2.5-3B (fp16), takes the judge-filtered
contrastive pairs (`results/judge_scores_qwen2.5-3b.csv`, score >= 60), collects
residual-stream activations and saves per-layer `mean(pos) - mean(neg)` vectors.

**Enable GPU** for this notebook. Output `.pt` files land under
`emotion_vectors/Qwen2.5-3B-Instruct/` and are fetched as kernel output.

In [ ]:
!git clone --branch emotion-encoder --depth 1 https://github.com/dmagog/persona-emotions.git
%cd persona-emotions

In [ ]:
!pip install -q -U transformers accelerate

In [ ]:
!python -m emotion.extract_vectors \
    --model_name Qwen/Qwen2.5-3B-Instruct \
    --data-dir eval_emotion/Qwen2.5-3B-Instruct \
    --judge-scores results/judge_scores_qwen2.5-3b.csv \
    --threshold 60 \
    --save-dir emotion_vectors/Qwen2.5-3B-Instruct \
    --emotion all

In [ ]:
# Quick sanity: shapes + pairwise cosine similarity between emotion vectors
import torch, glob, os
vecs = {}
for f in sorted(glob.glob('emotion_vectors/Qwen2.5-3B-Instruct/*_response_avg_diff.pt')):
    emo = os.path.basename(f).split('_response')[0]
    vecs[emo] = torch.load(f)
    print(emo, tuple(vecs[emo].shape))
if vecs:
    layer = list(vecs.values())[0].shape[0] // 2
    import itertools
    names = list(vecs)
    print('\ncosine @ layer', layer)
    for a, b in itertools.combinations(names, 2):
        va, vb = vecs[a][layer], vecs[b][layer]
        cos = torch.nn.functional.cosine_similarity(va, vb, dim=0).item()
        print(f'{a:>8} vs {b:<8} {cos:+.3f}')